# 🎬 Auto Video Studio — Formula DNA + OmniVoice + Pexels

**Run cells 1 → 2 → 3 in order.**  
Cell 3 launches the full Gradio UI with a public URL.

**Before running:**  
- Runtime → Change runtime type → **T4 GPU**  
- Get a free Gemini API key at [aistudio.google.com](https://aistudio.google.com)  
- Get a free Pexels API key at [pexels.com/api](https://www.pexels.com/api/)  
- Have your Writing Formula ready (same formula from your video editor system)  
- Have a 3–10s reference voice audio clip (.wav or .mp3)

## Cell 1 — Install Dependencies

In [ ]:
import subprocess, sys, os

print('📦 Installing dependencies...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'omnivoice', 'transformers>=5.3', 'gradio>=6.0',
    'google-genai', 'pydub', 'requests', 'scipy', 'soundfile', 'numpy',
], check=True)

result = subprocess.run(['ffmpeg', '-version'], capture_output=True)
if result.returncode == 0:
    print('✅ ffmpeg found')
else:
    subprocess.run(['apt-get', 'install', '-y', '-q', 'ffmpeg'], check=True)
    print('✅ ffmpeg installed')

PROJ = '/content/video_project'
for d in ['audio', 'stock_videos', 'assets', 'subtitles', 'output', 'cache']:
    os.makedirs(f'{PROJ}/{d}', exist_ok=True)

print('✅ All dependencies installed')
print('✅ Project directories created at', PROJ)
print('\n➡️  Run Cell 2 to load OmniVoice model')


## Cell 2 — Load OmniVoice Model  *(T4 GPU required)*

In [ ]:
import torch, numpy as np

print('🔊 Loading OmniVoice model on T4 GPU...')
print(f'   CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'   GPU: {torch.cuda.get_device_name(0)}')
    print(f'   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

from omnivoice import OmniVoice, OmniVoiceGenerationConfig

OMNIVOICE_MODEL = OmniVoice.from_pretrained(
    'k2-fsa/OmniVoice',
    device_map='cuda',
    dtype=torch.float16,
    load_asr=True,
    token=False,
)
SAMPLING_RATE = OMNIVOICE_MODEL.sampling_rate

print(f'✅ OmniVoice loaded — sampling rate: {SAMPLING_RATE} Hz')
print('\n➡️  Run Cell 3 to launch the Gradio UI')


## Cell 3 — Pipeline Functions + Gradio UI

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  IMPORTS & SETUP
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import os, re, json, time, hashlib, shutil, subprocess, traceback
import concurrent.futures
from pathlib import Path
from urllib.parse import quote as url_quote
import requests
import numpy as np
import soundfile as sf
import gradio as gr
from google import genai as google_genai
from google.genai import types as genai_types

PROJ = '/content/video_project'
DNA_CACHE = Path(f'{PROJ}/cache')
DNA_CACHE.mkdir(exist_ok=True)
FORMULAS_STORE = DNA_CACHE / 'formulas.json'

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  GEMINI CLIENT WRAPPER (google-genai SDK)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def _gemini_call(api_key, prompt, model='gemini-2.5-flash',
                 temperature=0.7, max_tokens=32768, system=None):
    client = google_genai.Client(api_key=api_key)
    cfg_kwargs = {'temperature': temperature, 'max_output_tokens': max_tokens}
    if system:
        cfg_kwargs['system_instruction'] = system
    resp = client.models.generate_content(
        model=model,
        contents=prompt,
        config=genai_types.GenerateContentConfig(**cfg_kwargs),
    )
    return (resp.text or '').strip()

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  HELPERS
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def _formula_key(formula):
    return hashlib.md5(formula.encode()).hexdigest()

def _load_dna(formula):
    p = DNA_CACHE / f'{_formula_key(formula)}.json'
    if p.exists():
        try:
            return json.loads(p.read_text()).get('dna', '')
        except Exception:
            pass
    return ''

def _save_dna(formula, dna):
    p = DNA_CACHE / f'{_formula_key(formula)}.json'
    p.write_text(json.dumps({'dna': dna, 'len': len(formula)}))

def _xml(tag, text, fallback=''):
    m = re.search(rf'<{tag}>(.*?)</{tag}>', text, re.DOTALL | re.IGNORECASE)
    return m.group(1).strip() if m else fallback

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  FORMULA LIBRARY — named formula save/load
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def _load_saved_formulas():
    if FORMULAS_STORE.exists():
        try:
            return json.loads(FORMULAS_STORE.read_text())
        except Exception:
            pass
    return {}

def _save_named_formula(name, text):
    store = _load_saved_formulas()
    store[name] = text
    FORMULAS_STORE.write_text(json.dumps(store, ensure_ascii=False, indent=2))
    return sorted(store.keys())

def _get_formula_names():
    return sorted(_load_saved_formulas().keys())

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  FORMULA EXTRACTOR — PASS A / B / C
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

FORMULA_MODEL        = 'gemini-2.5-pro'
MAX_SCRIPT_CHARS     = 250_000
MAX_CHUNK_CHARS      = 80_000
TARGET_FORMULA_CHARS = 30_000
MAX_EXPANSION_PASSES = 3

ANALYSIS_PROMPT = '''You are a world-class script analyst. Reverse-engineer the exact writing formula
used in the script chunk below. Extract EVERY observable pattern — no guessing,
no vague statements. Anchor every observation with a short quote from the script.

Language of the script: {language}

=== SCRIPT CHUNK {index}/{total} ===
{chunk}
=== END CHUNK ===

Return a detailed technical report with these sections (be exhaustive, include direct quotes):

1. HOOK & OPENING PATTERNS — How does the writer open? List every hook with quotes.
2. SENTENCE RHYTHM & CADENCE — Avg length. Short/long pattern. Punchlines.
3. VOICE, TONE & PERSONA — Who speaks? Confidence? Authority markers?
4. VOCABULARY & PHRASING SIGNATURES — Pet words, repeated constructions, metaphors.
   List the 30 most distinctive words/phrases with quotes.
5. PARAGRAPH / SECTION STRUCTURE — How ideas chain. Topic→payload→echo? Claim→proof→payoff?
6. TRANSITIONS & CONNECTORS — Exact bridge words/sentences used. List them.
7. RETENTION TECHNIQUES — Open loops, cliffhangers, "but first..." delays. Quote each.
8. ESCALATION & TENSION BUILDING — How stakes/intensity grow.
9. EVIDENCE & SPECIFICITY — Numbers, names, dates, examples — density + style.
10. EMOTIONAL TRIGGERS — Fear, curiosity, aspiration, anger — which beats use which?
11. CLOSING & CTA PATTERNS — Endings, CTAs, bypasses.
12. FORBIDDEN / NEVER-USED PATTERNS — What does the writer AVOID?
13. ANY OTHER DISTINCTIVE SIGNATURES — Anything that makes this script unmistakable.

Be relentless. Quote the script. Output RAW PATTERNS ONLY — do not write a formula yet.'''

SYNTHESIS_PROMPT = '''You are building the definitive Writing Guidelines document for a Script Writer AI.
Below are {n_dumps} pattern reports from different chunks of the SAME script.
Merge them into ONE complete reusable FORMULA.

Target language: {language}
Niche name: {name}

=== RAW PATTERN DUMPS ===
{dumps}
=== END DUMPS ===

OUTPUT REQUIREMENTS
- Minimum length: {target_chars:,} characters of actionable instructions
- Tone: strict, imperative, prescriptive ("DO / NEVER / ALWAYS / EXAMPLE")
- Every rule concrete with example. NO vague advice.

MANDATORY SECTIONS (in this exact order, each with depth):

═══════════════════════════════════════════════════════════
FORMULA: {name}
═══════════════════════════════════════════════════════════

SECTION 1 — CORE IDENTITY & VOICE (narrator, register, persona anchors)
SECTION 2 — UNIVERSAL LAWS (15+ numbered laws, each: RULE → WHY → EXAMPLE → ANTI-EXAMPLE)
SECTION 3 — HOOK MECHANICS (all opening archetypes, templates, forbidden openings)
SECTION 4 — SENTENCE & CADENCE RULES (length ranges, rhythm, punchline placement)
SECTION 5 — PARAGRAPH / BEAT STRUCTURE (claim→proof→payoff, beat length, chaining)
SECTION 6 — TRANSITIONS & CONNECTORS (allowed bridges, forbidden transitions)
SECTION 7 — VOCABULARY & SIGNATURE PHRASES (60+ approved words, signature templates,
            forbidden words/phrases — ALL generic-AI sludge)
SECTION 8 — RETENTION & OPEN-LOOP SYSTEM (every retention device, density rules)
SECTION 9 — ESCALATION CURVE (intensity rise across script, milestones)
SECTION 10 — EVIDENCE & SPECIFICITY RULES (number/name/date density, insertion style)
SECTION 11 — EMOTIONAL TRIGGER PLAYBOOK (which emotion for which beat, language presets)
SECTION 12 — CLOSING & BYPASS PATTERNS (objection killers, CTA templates, final lines)
SECTION 13 — STYLE LOCK / ANTI-GENERIC FIREWALL (banned generic-AI words/phrases)
SECTION 14 — CHUNK-POSITION RULES (opening/early-mid/late-mid/closing chunk rules)
SECTION 15 — QUALITY CHECKLIST (12+ yes/no questions writer must pass before output)

NOW: produce the full formula. Be exhaustive. Quote where useful. Hit the {target_chars:,}-char target. Begin:'''

EXPANSION_PROMPT = '''Your formula below is only {current_chars:,} characters. We need at least
{target_chars:,} characters of ACTIONABLE content. Do NOT pad with filler.

Expand every thin section with:
  - More concrete examples drawn from the source style
  - More numbered laws (each: RULE → WHY → EXAMPLE → ANTI-EXAMPLE)
  - More entries in the vocabulary bank
  - More transition phrases, more hook templates
  - More forbidden-list items

Preserve the existing structure and all already-written content. Only ADD.

=== CURRENT FORMULA ===
{current}
=== END ===

Output the ENTIRE expanded formula (do not truncate). Target: {target_chars:,}+ characters.'''

def _split_script(text, max_chars=MAX_CHUNK_CHARS):
    if len(text) <= max_chars:
        return [text]
    paras = text.split('\n\n')
    chunks, current = [], ''
    for p in paras:
        if len(current) + len(p) + 2 <= max_chars:
            current = f'{current}\n\n{p}' if current else p
        else:
            if current:
                chunks.append(current)
            current = p
    if current:
        chunks.append(current)
    return chunks

def _gemini_call_retry(api_key, prompt, retries=3, **kwargs):
    last_err = None
    for attempt in range(1, retries + 1):
        try:
            text = _gemini_call(api_key, prompt, **kwargs)
            if len(text) < 200:
                raise RuntimeError(f'Gemini returned short output ({len(text)} chars)')
            return text
        except Exception as e:
            last_err = e
            wait = min(8, 2 * attempt)
            print(f'   ⚠️  Gemini error attempt {attempt}/{retries}: {e} — retry in {wait}s')
            time.sleep(wait)
    raise RuntimeError(f'Gemini failed after {retries} attempts: {last_err}')

def extract_formula_from_script(api_key, script, name, language='English', progress=None):
    script = (script or '').strip()
    if len(script) < 500:
        raise ValueError('Script too short to extract a formula (min 500 chars)')
    if len(script) > MAX_SCRIPT_CHARS:
        raise ValueError(f'Script too long: {len(script):,} chars (max {MAX_SCRIPT_CHARS:,})')

    chunks = _split_script(script)
    print(f'\n🧬 Extracting formula from {len(script):,}-char script ({len(chunks)} chunks)...')

    dumps = []
    for i, ch in enumerate(chunks, 1):
        if progress:
            progress(0.05 + 0.40 * (i - 1) / max(1, len(chunks)),
                     desc=f'PASS A: Analyzing chunk {i}/{len(chunks)}...')
        print(f'   📊 PASS A — analyzing chunk {i}/{len(chunks)}...')
        dump = _gemini_call_retry(
            api_key,
            ANALYSIS_PROMPT.format(chunk=ch, index=i, total=len(chunks), language=language),
            model=FORMULA_MODEL, temperature=0.35, max_tokens=65000,
        )
        dumps.append(dump)

    if progress:
        progress(0.55, desc='PASS B: Synthesizing unified formula...')
    print(f'   🔗 PASS B — synthesizing {len(dumps)} pattern dumps into formula...')
    merged = '\n\n'.join(f'--- CHUNK {i+1} ---\n{d}' for i, d in enumerate(dumps))
    formula = _gemini_call_retry(
        api_key,
        SYNTHESIS_PROMPT.format(
            dumps=merged, n_dumps=len(dumps), language=language,
            name=name, target_chars=TARGET_FORMULA_CHARS,
        ),
        model=FORMULA_MODEL, temperature=0.35, max_tokens=65000,
    )

    passes = 0
    while len(formula) < TARGET_FORMULA_CHARS and passes < MAX_EXPANSION_PASSES:
        passes += 1
        if progress:
            progress(0.70 + passes * 0.08,
                     desc=f'PASS C: Expanding formula ({len(formula):,} → {TARGET_FORMULA_CHARS:,})...')
        print(f'   📈 PASS C #{passes} — expanding from {len(formula):,} chars...')
        formula = _gemini_call_retry(
            api_key,
            EXPANSION_PROMPT.format(
                current=formula, current_chars=len(formula),
                target_chars=TARGET_FORMULA_CHARS,
            ),
            model=FORMULA_MODEL, temperature=0.35, max_tokens=65000,
        )

    if progress:
        progress(1.0, desc='Done!')
    print(f'\n🎉 Formula extracted: {len(formula):,} chars')
    return formula

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  PHASE 1 — FORMULA DNA EXTRACTION
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def extract_formula_dna(api_key, formula, title, num_chunks, language='English'):
    cached_dna = _load_dna(formula)

    outline_fmt = ''
    for i in range(1, num_chunks + 1):
        role = 'HOOK/OPENING' if i == 1 else ('CLOSING/CTA' if i == num_chunks else 'BODY')
        closing = (
            'CLOSING RULES: Build toward CTA only. No new tangents. '
            'Deliver the title climax. Copy CTA phrase verbatim. Script feels 100% complete.\n'
            if i == num_chunks else ''
        )
        outline_fmt += (
            f'<chunk_{i}_outline>\n'
            f'STEP-BY-STEP recipe for chunk {i} ({role}) — 100% specific to "{title}".\n'
            f'Each step: STEP N | WHAT TO WRITE | FORMULA TECHNIQUE | MANDATORY PHRASE\n'
            f'{closing}'
            f'</chunk_{i}_outline>\n\n'
            f'<chunk_{i}_keyword>1-3 word Pexels stock video search keyword for chunk {i}</chunk_{i}_keyword>\n\n'
        )

    if cached_dna and len(cached_dna) >= 1500:
        prompt = (
            f'You are a script outline generator.\n'
            f'Use the Formula DNA below to create chunk-specific writing recipes.\n\n'
            f'FORMULA DNA:\n{cached_dna}\n\n'
            f'VIDEO TITLE: "{title}"\nLANGUAGE: {language}\nCHUNKS: {num_chunks}\n\n'
            f'Output ONLY the XML below. No markdown, no code blocks.\n\n'
            f'<anchor>main subject of this video title</anchor>\n'
            f'<cta>exact CTA from DNA verbatim</cta>\n\n'
            f'{outline_fmt}'
        )
        text = _gemini_call(api_key, prompt, temperature=0.1, max_tokens=16384)
        dna = cached_dna
        print('   ✅ Formula DNA loaded from disk cache (fast path)')
    else:
        dna_template = (
            '## 1. OPENING LAWS\n[Copy exact opening phrases/hooks verbatim.]\n\n'
            '## 2. TONE & VOICE LAWS\n[Exact tone, register, energy, person.]\n\n'
            '## 3. STRUCTURE LAWS\n[Paragraph/sentence rules. Section lengths.]\n\n'
            '## 4. MANDATORY PHRASES & HOOKS\n[EVERY phrase that MUST appear — verbatim.]\n\n'
            '## 5. FORBIDDEN WORDS & PATTERNS\n[Every banned word/technique.]\n\n'
            '## 6. STORYTELLING & CONTENT LAWS\n[Story structure, examples, social proof.]\n\n'
            '## 7. ENGAGEMENT & RETENTION LAWS\n[Hooks, cliffhangers, pattern interrupts.]\n\n'
            '## 8. PROMO & CTA LAWS\n[Promo count, placement, exact CTA verbatim.]\n\n'
            '## 9. LANGUAGE & STYLE LAWS\n[Vocabulary level, contractions, idioms.]\n\n'
            '## 10. NICHE-SPECIFIC LAWS\n[All domain-specific rules unique to this formula.]'
        )
        prompt = (
            f'You are an elite YouTube script production analyst.\n'
            f'Do TWO things:\n'
            f'1. EXTRACT COMPLETE FORMULA DNA from the Writing Guidelines.\n'
            f'2. CREATE PRESCRIPTIVE SCRIPT OUTLINES — step-by-step per chunk.\n\n'
            f'VIDEO TITLE: "{title}"\nLANGUAGE: {language}\nCHUNKS: {num_chunks}\n\n'
            f'<writing_guidelines>\n{formula}\n</writing_guidelines>\n\n'
            f'Output ONLY the XML below. No markdown, no code blocks.\n\n'
            f'<anchor>main subject of this video title</anchor>\n'
            f'<cta>exact CTA from Writing Guidelines verbatim</cta>\n\n'
            f'<formula_dna>\n'
            f'CRITICAL: Extract EVERY law. Be EXHAUSTIVE. Miss nothing.\n\n'
            f'{dna_template}\n'
            f'</formula_dna>\n\n'
            f'{outline_fmt}'
        )
        print('   📤 Extracting formula DNA + outlines (~20-30s)...')
        text = _gemini_call(api_key, prompt, temperature=0.1, max_tokens=65536)
        raw_dna = _xml('formula_dna', text)
        dna = raw_dna if len(raw_dna) >= 1500 else formula[:12000]
        if len(dna) >= 1500:
            _save_dna(formula, dna)
            print(f'   ✅ Formula DNA extracted ({len(dna):,} chars) — cached to disk')

    outlines, keywords = {}, {}
    for i in range(1, num_chunks + 1):
        m = re.search(rf'<chunk_{i}_outline>(.*?)</chunk_{i}_outline>', text, re.DOTALL | re.IGNORECASE)
        if m and len(m.group(1).strip()) > 50:
            outlines[i] = m.group(1).strip()
        kw = re.search(rf'<chunk_{i}_keyword>(.*?)</chunk_{i}_keyword>', text, re.DOTALL | re.IGNORECASE)
        keywords[i] = kw.group(1).strip() if kw else 'finance trading'

    return {
        'dna'     : dna,
        'outlines': outlines,
        'keywords': keywords,
        'anchor'  : _xml('anchor', text, title),
        'cta'     : _xml('cta', text, 'Subscribe and hit the bell'),
    }

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  PHASE 2 — WRITE ONE CHUNK
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def write_chunk(api_key, title, language, formula, plan,
                chunk_idx, total_chunks, target_chars, previous_context=''):
    dna      = plan['dna']
    outline  = plan['outlines'].get(chunk_idx, f'Write part {chunk_idx} of "{title}".')
    is_hook  = chunk_idx == 1
    is_final = chunk_idx == total_chunks
    role     = 'HOOK / OPENING' if is_hook else ('CLOSING / CTA' if is_final else 'BODY')

    formula_block = f'\n<raw_writing_guidelines>\n{formula}\n</raw_writing_guidelines>\n' if len(formula) <= 65000 else ''
    continuation  = (f'\nCONTINUATION — last 500 chars of previous chunk:\n"{previous_context}"\n'
                     if previous_context else '')

    flow_check = (
        'Hook grabs attention in first 3 seconds' if is_hook
        else 'CTA placed exactly as formula requires' if is_final
        else 'Smooth natural continuation from previous chunk'
    )

    system = (
        'You are an elite YouTube scriptwriter. '
        'Execute the writing recipe step by step. '
        'Every sentence must execute a specific formula rule. '
        'Output ONLY the script text — no commentary, no headings, no chunk labels.'
    )

    prompt = (
        f'FORMULA DNA (ALL NICHE LAWS):\n{dna}\n'
        f'{formula_block}'
        f'\nVIDEO TITLE: "{title}"\nLANGUAGE: {language}\n'
        f'CHUNK: {chunk_idx} of {total_chunks} ({role})\n'
        f'TARGET LENGTH: {target_chars:,} characters\n'
        f'{continuation}'
        f'\nSTEP-BY-STEP RECIPE:\n{outline}\n\n'
        f'══ SELF-CHECK ══\n'
        f'✓ Every sentence executes a formula rule\n'
        f'✓ Mandatory phrases used verbatim\n'
        f'✓ No forbidden words/patterns\n'
        f'✓ Tone matches formula exactly\n'
        f'✓ {flow_check}\n'
        f'✓ Length ~{target_chars:,} characters\n'
        f'═══════════════\n\n'
        f'Write chunk {chunk_idx} script text NOW:'
    )

    max_tokens = min(65536, max(int(target_chars / 2) + 6000, 12000))
    temp = 0.3 if is_hook else (0.2 if is_final else 0.7)

    return _gemini_call(api_key, prompt, model='gemini-2.5-flash',
                        temperature=temp, max_tokens=max_tokens, system=system)

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  KEYWORD REFINEMENT — based on actual written chunk text
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def _generate_better_keywords(api_key, chunks):
    chunk_lines = '\n'.join(
        f'CHUNK {c["chunk_id"]}: {c["text"][:300].strip()}...'
        for c in chunks
    )
    kw_tags = '\n'.join(
        f'<kw_{c["chunk_id"]}>specific visual keyword</kw_{c["chunk_id"]}>'
        for c in chunks
    )
    prompt = (
        'Generate the most specific, visually accurate Pexels stock video search keyword for each chunk.\n'
        'The keyword MUST be tied to the EXACT subject of that chunk — not the general niche.\n\n'
        'BAD (too generic): "finance money", "business success", "motivational speaker"\n'
        'GOOD (specific scene): "Warren Buffett press conference", "stock market crash graph", '
        '"bank vault opening", "NYSE trading floor", "gold bars stacked", "compound interest chart"\n\n'
        f'CHUNKS:\n{chunk_lines}\n\n'
        f'Output ONLY XML:\n{kw_tags}'
    )
    try:
        resp = _gemini_call(api_key, prompt, model='gemini-2.5-flash', temperature=0.1, max_tokens=1000)
        updated = 0
        for c in chunks:
            m = re.search(rf'<kw_{c["chunk_id"]}>(.*?)</kw_{c["chunk_id"]}>', resp, re.DOTALL | re.IGNORECASE)
            if m:
                kw = m.group(1).strip()
                if kw and len(kw) > 2:
                    c['keyword'] = kw
                    updated += 1
        print(f'   🎯 Keywords refined for {updated}/{len(chunks)} chunks')
    except Exception as e:
        print(f'   ⚠️  Keyword refinement skipped: {e}')
    return chunks

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  FULL SCRIPT GENERATION — PARALLEL (fix #4: speed)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def generate_full_script(api_key, title, formula, num_chunks,
                         language='English', chars_per_chunk=1100, progress=None):
    if progress:
        progress(0.05, desc='Phase 1: Extracting formula DNA...')
    plan = extract_formula_dna(api_key, formula, title, num_chunks, language)
    print(f'\n✅ Phase 1 done — anchor: "{plan["anchor"]}"')

    print(f'\n✍️  Writing {num_chunks} chunks in parallel (up to 5 concurrent)...')
    if progress:
        progress(0.12, desc=f'Phase 2: Writing {num_chunks} chunks in parallel...')

    chunks_out  = [None] * num_chunks
    done_count  = [0]

    def _write_one(i):
        text = write_chunk(
            api_key=api_key, title=title, language=language, formula=formula,
            plan=plan, chunk_idx=i, total_chunks=num_chunks,
            target_chars=chars_per_chunk, previous_context='',
        )
        text = re.sub(r'=== END.*', '', text, flags=re.IGNORECASE).strip()
        kw = plan['keywords'].get(i, 'finance trading')
        print(f'   ✅ Chunk {i}: {len(text):,} chars — keyword: "{kw}"')
        return i, text, kw

    max_workers = min(num_chunks, 5)
    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(_write_one, i): i for i in range(1, num_chunks + 1)}
        for fut in concurrent.futures.as_completed(futures):
            try:
                i, text, kw = fut.result()
                chunks_out[i - 1] = {'chunk_id': i, 'text': text, 'keyword': kw}
                done_count[0] += 1
                if progress:
                    progress(0.12 + (done_count[0] / num_chunks) * 0.83,
                             desc=f'Phase 2: {done_count[0]}/{num_chunks} chunks done...')
            except Exception as e:
                print(f'   ❌ Chunk write error: {e}')

    chunks_out = [c for c in chunks_out if c is not None]
    total = sum(len(c['text']) for c in chunks_out)
    print(f'\n🎉 Script done: {len(chunks_out)} chunks, {total:,} chars')

    print('\n🎯 Refining Pexels keywords from actual script content...')
    if progress:
        progress(0.97, desc='Refining video keywords...')
    chunks_out = _generate_better_keywords(api_key, chunks_out)
    return chunks_out

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  VOICE CLONE — SAVE / LOAD / REUSE
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

VOICE_PROFILE_PATH = f'{PROJ}/assets/voice_profile.json'
_voice_clone_prompt_cache = None

def clone_and_save_voice(ref_audio_path):
    global _voice_clone_prompt_cache
    if not ref_audio_path or not os.path.exists(ref_audio_path):
        return '❌ No audio file provided'
    ref_dest = f'{PROJ}/assets/voice_reference.wav'
    shutil.copy2(ref_audio_path, ref_dest)
    _voice_clone_prompt_cache = OMNIVOICE_MODEL.create_voice_clone_prompt(ref_audio=ref_dest)
    with open(VOICE_PROFILE_PATH, 'w') as f:
        json.dump({'ref_audio': ref_dest}, f)
    return '✅ Voice cloned and saved — reused for all audio generation automatically'

def _ensure_voice_loaded():
    global _voice_clone_prompt_cache
    if _voice_clone_prompt_cache is not None:
        return True
    if os.path.exists(VOICE_PROFILE_PATH):
        with open(VOICE_PROFILE_PATH) as f:
            src = json.load(f).get('ref_audio', '')
        if src and os.path.exists(src):
            _voice_clone_prompt_cache = OMNIVOICE_MODEL.create_voice_clone_prompt(ref_audio=src)
            print('✅ Voice profile reloaded from saved reference')
            return True
    return False

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  TTS — FIXED AUDIO FORMAT (fix #6)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def generate_chunk_audio(chunk_text, chunk_idx, language='Auto'):
    if not _ensure_voice_loaded():
        raise ValueError('Voice not cloned — clone a voice first in Settings tab')
    lang_map = {'Auto': 'auto', 'English': 'en', 'French': 'fr', 'Spanish': 'es',
                'German': 'de', 'Arabic': 'ar', 'Portuguese': 'pt', 'Italian': 'it'}
    lang_code = lang_map.get(language, 'auto')

    gen_config = OmniVoiceGenerationConfig(
        num_step=16, guidance_scale=3.0, denoise=0.8,
        preprocess_prompt=True, postprocess_output=True,
    )
    audio = OMNIVOICE_MODEL.generate(
        text=chunk_text, language=lang_code,
        generation_config=gen_config,
        voice_clone_prompt=_voice_clone_prompt_cache,
    )

    # Robust conversion: handle torch tensors, float16, wrong shapes
    if hasattr(audio, 'cpu'):
        audio = audio.cpu().numpy()
    elif hasattr(audio, 'numpy'):
        audio = audio.numpy()
    audio = np.asarray(audio, dtype=np.float32)
    if audio.ndim > 1:
        audio = audio.squeeze()
    audio = np.clip(audio, -1.0, 1.0)

    out_path = f'{PROJ}/audio/chunk_{chunk_idx:02d}.wav'
    sf.write(out_path, audio, SAMPLING_RATE)
    duration = len(audio) / SAMPLING_RATE
    print(f'   🎤 Chunk {chunk_idx}: {duration:.1f}s')
    return out_path, duration

def generate_all_audio(chunks, language='Auto', progress=None):
    results = []
    for i, chunk in enumerate(chunks):
        if progress:
            progress(i / len(chunks), desc=f'TTS chunk {i+1}/{len(chunks)}...')
        path, dur = generate_chunk_audio(chunk['text'], chunk['chunk_id'], language)
        results.append({**chunk, 'audio_path': path, 'duration': dur})
    return results

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  WHISPER — WORD-LEVEL TIMESTAMPS
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def get_word_timestamps(audio_path, text):
    try:
        result = OMNIVOICE_MODEL.transcribe(audio_path, word_timestamps=True)
        word_times = []
        for seg in result.get('segments', []):
            for w in seg.get('words', []):
                word_times.append({'word': w['word'].strip(), 'start': w['start'], 'end': w['end']})
        if word_times:
            return word_times
    except Exception as e:
        print(f'   ASR fallback (uniform): {e}')

    words = text.split()
    data, sr = sf.read(audio_path)
    total_dur = len(data) / sr
    spw = total_dur / max(len(words), 1)
    return [{'word': w, 'start': i * spw, 'end': (i + 1) * spw} for i, w in enumerate(words)]

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  ASS SUBTITLE BUILDER — KARAOKE WORD HIGHLIGHTING
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def build_ass_subtitles(chunks_with_audio):
    ass_path = f'{PROJ}/subtitles/captions.ass'
    header = (
        '[Script Info]\nScriptType: v4.00+\nWrapStyle: 0\nScaledBorderAndShadow: yes\n'
        'PlayResX: 1920\nPlayResY: 1080\n\n'
        '[V4+ Styles]\n'
        'Format: Name, Fontname, Fontsize, PrimaryColour, SecondaryColour, OutlineColour, BackColour, '
        'Bold, Italic, Underline, StrikeOut, ScaleX, ScaleY, Spacing, Angle, BorderStyle, Outline, Shadow, '
        'Alignment, MarginL, MarginR, MarginV, Encoding\n'
        'Style: Default,Arial,54,&H00FFFFFF,&H000000FF,&H00000000,&H90000000,-1,0,0,0,100,100,0,0,1,3,2,'
        '5,900,80,120,1\n\n'
        '[Events]\n'
        'Format: Layer, Start, End, Style, Name, MarginL, MarginR, MarginV, Effect, Text\n'
    )

    def ts(sec):
        h = int(sec // 3600); m = int((sec % 3600) // 60)
        s = int(sec % 60); cs = int((sec % 1) * 100)
        return f'{h}:{m:02d}:{s:02d}.{cs:02d}'

    lines = []
    global_offset = 0.0

    for chunk in chunks_with_audio:
        audio_path = chunk.get('audio_path', '')
        duration   = chunk.get('duration', 0)
        text       = chunk.get('text', '')
        if not audio_path or not os.path.exists(audio_path):
            global_offset += duration
            continue

        wt = get_word_timestamps(audio_path, text)
        WORDS_PER_LINE = 7
        groups = [wt[i:i+WORDS_PER_LINE] for i in range(0, len(wt), WORDS_PER_LINE)]

        for group in groups:
            if not group:
                continue
            for wi, wdata in enumerate(group):
                w_start = wdata['start'] + global_offset
                w_end   = wdata['end']   + global_offset
                parts = []
                for j, wd in enumerate(group):
                    wclean = re.sub(r'[{}\\]', '', wd['word'])
                    if j == wi:
                        parts.append(r'{\c&H00CFFF&\b1}' + wclean + r'{\c&HFFFFFF&}')
                    else:
                        parts.append(wclean)
                lines.append(f'Dialogue: 0,{ts(w_start)},{ts(w_end)},Default,,0,0,0,,{" ".join(parts)}')
        global_offset += duration

    with open(ass_path, 'w', encoding='utf-8') as f:
        f.write(header + '\n'.join(lines))
    return ass_path

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  PEXELS — 10 VIDEOS PER CHUNK (fix #5)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def download_10_pexels_videos(api_key, keyword, chunk_id):
    headers  = {'Authorization': api_key}
    out_dir  = Path(f'{PROJ}/stock_videos/chunk_{chunk_id:02d}')
    out_dir.mkdir(parents=True, exist_ok=True)
    fallbacks = [keyword, 'finance trading', 'stock market chart', 'business money', 'abstract technology']
    candidates = []

    for kw in fallbacks:
        if len(candidates) >= 10:
            break
        try:
            url = (f'https://api.pexels.com/videos/search?query={url_quote(kw)}'
                   f'&orientation=landscape&per_page=15&size=large')
            resp = requests.get(url, headers=headers, timeout=20)
            if resp.status_code != 200:
                continue
            for vid in resp.json().get('videos', []):
                if len(candidates) >= 10:
                    break
                best_f, best_s = None, -1
                for vf in vid.get('video_files', []):
                    w, h = vf.get('width', 0), vf.get('height', 0)
                    if w < 1280 or h < 720:
                        continue
                    s = w * h + (500000 if abs(w - 1920) < 200 else 0)
                    if vf.get('quality') in ('hd', 'full_hd'):
                        s += 300000
                    if s > best_s:
                        best_s = s; best_f = vf
                if best_f:
                    candidates.append(best_f)
        except Exception as e:
            print(f'   Pexels search "{kw}": {e}')

    downloaded = []
    for vi, vf in enumerate(candidates):
        out_path = out_dir / f'v{vi:02d}.mp4'
        try:
            dl = requests.get(vf['link'], stream=True, timeout=120)
            with open(out_path, 'wb') as f:
                for data in dl.iter_content(chunk_size=65536):
                    f.write(data)
            downloaded.append(str(out_path))
            print(f'   📹 Chunk {chunk_id} v{vi+1}/{len(candidates)}: {vf.get("width")}×{vf.get("height")}')
        except Exception as e:
            print(f'   Download error v{vi}: {e}')

    if not downloaded:
        print(f'   ⚠️  No videos for chunk {chunk_id} ("{keyword}")')
    return downloaded

def download_all_stock_videos(pexels_key, chunks, progress=None):
    results = []
    for i, chunk in enumerate(chunks):
        if progress:
            progress(i / len(chunks),
                     desc=f'Downloading 10 videos for chunk {i+1}/{len(chunks)}: "{chunk["keyword"]}"...')
        paths = download_10_pexels_videos(pexels_key, chunk['keyword'], chunk['chunk_id'])
        results.append({**chunk, 'video_paths': paths})
        time.sleep(0.5)
    return results

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  FFMPEG — MULTI-CLIP BACKGROUND + RENDERER
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def _vid_duration(path):
    try:
        r = subprocess.run(
            ['ffprobe', '-v', 'error', '-show_entries', 'format=duration',
             '-of', 'default=noprint_wrappers=1:nokey=1', path],
            capture_output=True, text=True)
        return float(r.stdout.strip())
    except Exception:
        return 10.0

def _make_multi_bg(video_paths, duration, out_path):
    n = len(video_paths)
    if n == 1:
        return video_paths[0]
    clip_dur = max(2.0, duration / n)
    temp_clips = []
    for vi, vp in enumerate(video_paths):
        clip_out = out_path.replace('.mp4', f'_c{vi}.mp4')
        loops = max(1, int(np.ceil(clip_dur / max(_vid_duration(vp), 0.1))))
        r = subprocess.run([
            'ffmpeg', '-y', '-stream_loop', str(loops - 1), '-i', vp,
            '-vf', (f'scale=1920:1080:force_original_aspect_ratio=increase,'
                    f'crop=1920:1080,trim=duration={clip_dur},setpts=PTS-STARTPTS'),
            '-an', '-r', '30', '-c:v', 'libx264', '-preset', 'ultrafast', '-crf', '28', clip_out
        ], capture_output=True)
        if r.returncode == 0 and os.path.exists(clip_out):
            temp_clips.append(clip_out)
    if not temp_clips:
        return video_paths[0]
    concat_txt = out_path.replace('.mp4', '_c.txt')
    with open(concat_txt, 'w') as f:
        f.write('\n'.join(f"file '{p}'" for p in temp_clips))
    r = subprocess.run([
        'ffmpeg', '-y', '-f', 'concat', '-safe', '0',
        '-i', concat_txt, '-c', 'copy', out_path
    ], capture_output=True)
    for p in temp_clips:
        try: os.remove(p)
        except: pass
    return out_path if (r.returncode == 0 and os.path.exists(out_path)) else video_paths[0]

def _render_segment(chunk, seg_out, ass_path, character_image):
    audio_path  = chunk.get('audio_path', '')
    video_paths = chunk.get('video_paths') or []
    if not video_paths and chunk.get('video_path'):
        video_paths = [chunk['video_path']]
    duration = chunk.get('duration', 5.0)
    idx      = chunk.get('chunk_id', 0)

    if not audio_path or not os.path.exists(audio_path):
        print(f'  ⚠️  Chunk {idx}: no audio — skipping')
        return False

    video_paths = [p for p in video_paths if p and os.path.exists(p)]
    has_char    = bool(character_image and os.path.exists(character_image))

    # Combine multiple clips into one bg video
    if len(video_paths) > 1:
        tmp_bg     = f'{PROJ}/output/tmp_bg_{idx:02d}.mp4'
        video_path = _make_multi_bg(video_paths, duration, tmp_bg)
    elif video_paths:
        video_path = video_paths[0]
    else:
        video_path = ''

    has_video = bool(video_path and os.path.exists(video_path))
    loops     = max(1, int(np.ceil(duration / max(_vid_duration(video_path), 0.1)))) if has_video else 1

    if has_video:
        bg = (f'[0:v]scale=1920:1080:force_original_aspect_ratio=increase,'
              f'crop=1920:1080,trim=duration={duration},setpts=PTS-STARTPTS[bg];'
              f'color=black:s=1920x1080:r=30[blk];'
              f'[bg][blk]blend=all_mode=normal:all_opacity=0.55[darkbg]')
        vid_args  = ['-stream_loop', str(loops - 1), '-i', video_path]
        audio_idx = 1 + (1 if has_char else 0)
    else:
        bg = f'color=black:s=1920x1080:r=30:d={duration}[darkbg]'
        vid_args  = []
        audio_idx = 0 + (1 if has_char else 0)

    char_args = ['-i', character_image] if has_char else []
    char_idx  = 1 if has_video else 0

    if has_char:
        vf = (f'{bg};'
              f'[{char_idx}:v]scale=-1:518:flags=lanczos[char];'
              f'[darkbg][char]overlay=x=30:y=(H-h)/2+60[withchar];'
              f'[withchar]ass={ass_path}[out]')
    else:
        vf = f'{bg};[darkbg]ass={ass_path}[out]'

    cmd = (
        ['ffmpeg', '-y'] + vid_args + char_args + ['-i', audio_path]
        + ['-filter_complex', vf]
        + ['-map', '[out]', '-map', f'{audio_idx}:a']
        + ['-t', str(duration)]
        + ['-c:v', 'libx264', '-preset', 'ultrafast', '-crf', '26', '-b:v', '2500k']
        + ['-c:a', 'aac', '-ar', '44100', '-b:a', '128k']
        + ['-r', '30', '-pix_fmt', 'yuv420p', seg_out]
    )
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        print(f'  ⚠️  Segment {idx} error:\n{r.stderr[-400:]}')
        return False
    print(f'  ✅ Segment {idx}: {duration:.1f}s ({len(video_paths)} clips)')
    return True

def render_final_video(chunks, character_image=None, bg_music=None, progress=None):
    if progress:
        progress(0.03, desc='Building karaoke captions...')
    ass_path = build_ass_subtitles(chunks)
    print(f'✅ ASS: {ass_path}')

    segment_paths = []
    for i, chunk in enumerate(chunks):
        if progress:
            progress(0.08 + (i / len(chunks)) * 0.72, desc=f'Segment {i+1}/{len(chunks)}...')
        seg_out = f'{PROJ}/output/segment_{i+1:02d}.mp4'
        if _render_segment(chunk, seg_out, ass_path, character_image):
            segment_paths.append(seg_out)

    if not segment_paths:
        raise ValueError('No segments rendered — check audio and stock video steps')

    if progress:
        progress(0.83, desc='Concatenating...')
    concat_list = f'{PROJ}/output/concat.txt'
    with open(concat_list, 'w') as f:
        for p in segment_paths:
            f.write(f"file '{p}'\n")

    joined = f'{PROJ}/output/joined.mp4'
    subprocess.run(['ffmpeg', '-y', '-f', 'concat', '-safe', '0',
                    '-i', concat_list, '-c', 'copy', joined],
                   check=True, capture_output=True)

    final_path = f'{PROJ}/output/final_video.mp4'
    if bg_music and os.path.exists(bg_music):
        if progress:
            progress(0.92, desc='Mixing background music...')
        subprocess.run([
            'ffmpeg', '-y', '-i', joined, '-stream_loop', '-1', '-i', bg_music,
            '-filter_complex',
            '[1:a]volume=0.12[bgm];[0:a][bgm]amix=inputs=2:duration=first:dropout_transition=3[outa]',
            '-map', '0:v', '-map', '[outa]', '-c:v', 'copy', '-c:a', 'aac', '-b:a', '192k', final_path,
        ], check=True, capture_output=True)
    else:
        shutil.copy2(joined, final_path)

    if progress:
        progress(1.0, desc='Done!')
    size_mb = os.path.getsize(final_path) / 1e6
    print(f'\n🎉 Final video: {final_path} ({size_mb:.1f} MB)')
    return final_path

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  SESSION STATE
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

_session = {'chunks': [], 'character_img': None, 'bg_music': None}

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  GRADIO HANDLERS
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def handle_clone_voice(ref_audio):
    if ref_audio is None:
        return '❌ Upload a voice reference audio file first'
    return clone_and_save_voice(ref_audio)

def handle_extract_formula(gemini_key, script, name, language, progress=gr.Progress()):
    if not gemini_key.strip():
        return '', '❌ Gemini API key required (Settings tab)', gr.Dropdown()
    if not script.strip() or len(script.strip()) < 500:
        return '', '❌ Paste a script (minimum 500 chars)', gr.Dropdown()
    if not name.strip():
        return '', '❌ Give the formula a name (e.g. "Buffett Finance Style")', gr.Dropdown()
    try:
        formula = extract_formula_from_script(
            api_key=gemini_key.strip(), script=script.strip(),
            name=name.strip(), language=language, progress=progress,
        )
        names = _save_named_formula(name.strip(), formula)
        return (
            formula,
            f'✅ Formula "{name.strip()}" saved ({len(formula):,} chars) — select it in Studio tab',
            gr.Dropdown(choices=names, value=name.strip()),
        )
    except Exception as e:
        return '', f'❌ {e}\n{traceback.format_exc()[-500:]}', gr.Dropdown()

def handle_formula_select(name):
    if not name:
        return '', ''
    store = _load_saved_formulas()
    text  = store.get(name, '')
    if not text:
        return '', f'❌ Formula "{name}" not found in library'
    return text, f'✅ Loaded: {name} ({len(text):,} chars)'

def handle_save_formula_manual(name, text):
    if not name.strip():
        return '❌ Enter a formula name first'
    if len(text.strip()) < 100:
        return '❌ Formula too short (min 100 chars)'
    _save_named_formula(name.strip(), text.strip())
    return f'✅ Saved "{name.strip()}" to formula library'

def handle_generate_script(gemini_key, active_formula_text, formula_box_text, num_chunks,
                            total_script_chars, title, language, progress=gr.Progress()):
    formula = (active_formula_text or formula_box_text or '').strip()
    if not gemini_key.strip():
        return None, '❌ Gemini API key required (Settings tab)'
    if len(formula) < 100:
        return None, '❌ Select a formula from the Studio dropdown, or paste one in Settings → Writing Formula'
    if not title.strip():
        return None, '❌ Enter a video title'
    n_chunks = int(num_chunks)
    chars_per_chunk = max(400, int(total_script_chars) // max(1, n_chunks))
    try:
        chunks = generate_full_script(
            api_key=gemini_key.strip(), title=title.strip(), formula=formula,
            num_chunks=n_chunks, language=language,
            chars_per_chunk=chars_per_chunk, progress=progress,
        )
        _session['chunks'] = chunks
        table = [[c['chunk_id'], c['keyword'],
                  c['text'][:180] + '...' if len(c['text']) > 180 else c['text']]
                 for c in chunks]
        total_chars = sum(len(c['text']) for c in chunks)
        return table, (f'✅ Script: {len(chunks)} chunks · {total_chars:,} chars '
                       f'(~{total_chars//5:,} words) · {chars_per_chunk:,} chars/chunk')
    except Exception as e:
        return None, f'❌ {e}\n{traceback.format_exc()[-500:]}'

def handle_generate_audio(language, progress=gr.Progress()):
    if not _session['chunks']:
        return '❌ Generate a script first (Step 1)'
    if not _ensure_voice_loaded():
        return '❌ Clone a voice first in Settings tab'
    try:
        updated = generate_all_audio(_session['chunks'], language, progress)
        _session['chunks'] = updated
        total_dur = sum(c.get('duration', 0) for c in updated)
        return f'✅ Audio: {len(updated)} chunks, {total_dur:.1f}s (~{total_dur/60:.1f} min)'
    except Exception as e:
        return f'❌ {e}'

def handle_download_videos(pexels_key, progress=gr.Progress()):
    if not _session['chunks']:
        return '❌ Generate a script first'
    if not pexels_key.strip():
        return '❌ Pexels API key required'
    try:
        updated = download_all_stock_videos(pexels_key.strip(), _session['chunks'], progress)
        _session['chunks'] = updated
        found = sum(1 for c in updated if c.get('video_paths'))
        total_vids = sum(len(c.get('video_paths', [])) for c in updated)
        return f'✅ Stock videos: {total_vids} files across {found}/{len(updated)} chunks'
    except Exception as e:
        return f'❌ {e}'

def handle_set_character(img_path):
    if img_path:
        dest = f'{PROJ}/assets/character.png'
        shutil.copy2(img_path, dest)
        _session['character_img'] = dest
        return '✅ Character image saved'
    return ''

def handle_set_bgmusic(audio_path):
    if audio_path:
        dest = f'{PROJ}/assets/bg_music.mp3'
        shutil.copy2(audio_path, dest)
        _session['bg_music'] = dest
        return '✅ Background music saved'
    return ''

def handle_render_video(progress=gr.Progress()):
    if not _session['chunks']:
        return None, '❌ Complete Steps 1–3 first'
    missing = [c['chunk_id'] for c in _session['chunks'] if not c.get('audio_path')]
    if missing:
        return None, f'❌ Missing audio for chunks: {missing} — run Step 2'
    try:
        path = render_final_video(
            chunks=_session['chunks'],
            character_image=_session.get('character_img'),
            bg_music=_session.get('bg_music'),
            progress=progress,
        )
        return path, f'✅ Final video: {os.path.getsize(path)/1e6:.1f} MB'
    except Exception as e:
        return None, f'❌ {e}\n{traceback.format_exc()[-600:]}'

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  GRADIO UI  (Gradio 6.x — theme/css go to launch())
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

CSS = '''
.step-label { font-size:17px; font-weight:700; color:#c4b5fd;
              border-left:4px solid #7c3aed; padding-left:10px; margin:10px 0 6px; }
footer { display:none !important; }
'''

with gr.Blocks(title='🎬 Auto Video Studio') as demo:

    active_formula = gr.State('')   # holds formula text from dropdown selection

    gr.Markdown(
        '# 🎬 Auto Video Studio\n'
        '**Formula DNA Script** → **OmniVoice TTS** → **Pexels Stock Video** → '
        '**Karaoke Captions** → **Final MP4**'
    )

    with gr.Tabs():

        # ── TAB 1: SETTINGS ──────────────────────────────────────────────
        with gr.Tab('⚙️  Settings'):
            gr.Markdown('### 🔑 API Keys')
            with gr.Row():
                gemini_key = gr.Textbox(label='Gemini API Key',
                                        placeholder='AIza...', type='password', scale=1)
                pexels_key = gr.Textbox(label='Pexels API Key',
                                        placeholder='Pexels API key...', type='password', scale=1)

            gr.Markdown('---\n### ✍️  Writing Formula  *(manual paste — or use Extract Formula tab)*')
            formula_box = gr.Textbox(
                label='Writing Formula / Writing Guidelines (manual override)',
                placeholder=(
                    'Optional: paste your formula here for manual use.\n'
                    'Tip: use the Extract Formula tab to auto-generate one, then select it from '
                    'the Studio tab dropdown.\n\n'
                    'DNA cached to disk — same formula on next run skips extraction (saves ~25s).'
                ),
                lines=10, max_lines=40,
            )
            with gr.Row():
                save_formula_name = gr.Textbox(
                    label='Save as (formula name)',
                    placeholder='e.g. Buffett Finance Style',
                    scale=2,
                )
                save_formula_btn = gr.Button('💾  Save to Library', variant='secondary', scale=1)
            save_formula_status = gr.Textbox(label='', interactive=False, lines=1)

            gr.Markdown('---\n### 🎤  Voice Clone  *(one-time per session)*')
            with gr.Row():
                voice_ref = gr.Audio(label='Voice Reference Audio (3-10s)', type='filepath', scale=2)
                clone_btn = gr.Button('🎤  Clone & Save Voice', variant='primary', scale=1)
            clone_status = gr.Textbox(label='Voice Status', interactive=False, lines=1)

            gr.Markdown('---\n### 🎭  Character & Background Music  *(optional)*')
            with gr.Row():
                char_img     = gr.Image(label='Character Image (PNG transparent bg)',
                                        type='filepath', scale=1)
                bg_music_inp = gr.Audio(label='Background Music (optional)',
                                        type='filepath', scale=1)
            with gr.Row():
                char_status = gr.Textbox(label='', interactive=False, lines=1)
                bgm_status  = gr.Textbox(label='', interactive=False, lines=1)

        # ── TAB 2: FORMULA EXTRACTOR ─────────────────────────────────────
        with gr.Tab('🧬  Extract Formula'):
            gr.Markdown(
                '### 🧬 Extract Formula From Existing Script\n'
                'Paste an existing script → Gemini reverse-engineers its complete writing formula.  \n'
                '**Pipeline:** PASS A (per-chunk patterns) → PASS B (synthesis) → PASS C (expand to 30K+ chars)  \n'
                'Uses **gemini-2.5-pro**. Takes ~2-5 min for long scripts.  \n'
                '**Formula is auto-saved** — select it from the Studio tab dropdown after extraction.'
            )
            with gr.Row():
                fe_name = gr.Textbox(
                    label='Formula Name',
                    placeholder='e.g. Buffett Finance Style, Tate Mindset, Mr Beast Pacing',
                    scale=2,
                )
                fe_lang = gr.Dropdown(
                    label='Language',
                    choices=['English', 'French', 'Spanish', 'German', 'Arabic', 'Portuguese', 'Italian'],
                    value='English', scale=1,
                )
            fe_script = gr.Textbox(
                label='Existing Script (500 chars min, 250K max)',
                placeholder='Paste a script you want to reverse-engineer the writing formula from...',
                lines=15, max_lines=40,
            )
            fe_btn    = gr.Button('🧬  Extract Formula', variant='primary')
            fe_status = gr.Textbox(label='Status', interactive=False, lines=2)
            fe_output = gr.Textbox(
                label='Extracted Formula (auto-saved — select from Studio dropdown)',
                lines=20, max_lines=60,
            )

        # ── TAB 3: STUDIO ────────────────────────────────────────────────
        with gr.Tab('🎬  Studio'):

            gr.Markdown('### 📚 Writing Formula')
            with gr.Row():
                formula_dropdown = gr.Dropdown(
                    label='Select Saved Formula',
                    choices=_get_formula_names(),
                    value=None,
                    scale=3,
                    interactive=True,
                )
                refresh_formula_btn = gr.Button('🔄  Refresh', variant='secondary', scale=0, min_width=90)
            formula_sel_status = gr.Textbox(label='', interactive=False, lines=1)

            gr.Markdown('### ⚙️  Script Settings')
            with gr.Row():
                num_chunks = gr.Slider(
                    label='Script Chunks  (scenes — each gets 10 stock videos)',
                    minimum=3, maximum=20, value=8, step=1, scale=1,
                )
                total_script_chars = gr.Slider(
                    label='Total Script Length',
                    minimum=10000, maximum=50000, value=20000, step=5000, scale=1,
                )
            gr.Markdown(
                '_Chars per chunk = Total ÷ Chunks.  '
                '20K ≈ 4,000 words ≈ 5-6 min video · 30K ≈ 6,000 words ≈ 8-9 min · 50K ≈ 10,000 words ≈ 14 min_'
            )

            gr.HTML('<hr style="margin:20px 0"/>')
            gr.HTML('<div class="step-label">① Generate Script  (Formula DNA + Gemini)</div>')
            with gr.Row():
                title_input = gr.Textbox(
                    label='Video Title',
                    placeholder="e.g. Warren Buffett's #1 Rule for Building Wealth in 2025",
                    scale=3,
                )
                lang_select = gr.Dropdown(
                    label='Language',
                    choices=['Auto', 'English', 'French', 'Spanish', 'German',
                             'Arabic', 'Portuguese', 'Italian'],
                    value='Auto', scale=1,
                )
            btn_script    = gr.Button('🧠  Generate Script', variant='primary')
            status_script = gr.Textbox(label='Status', interactive=False, lines=1)
            script_table  = gr.Dataframe(
                headers=['#', 'Pexels Keyword', 'Script Preview'],
                datatype=['number', 'str', 'str'],
                label='Script Chunks',
                wrap=True,
            )

            gr.HTML('<hr style="margin:20px 0"/>')
            gr.HTML('<div class="step-label">② Generate Voice Audio</div>')
            btn_audio    = gr.Button('🎤  Generate All Audio', variant='primary')
            status_audio = gr.Textbox(label='Status', interactive=False, lines=1)

            gr.HTML('<hr style="margin:20px 0"/>')
            gr.HTML('<div class="step-label">③ Download Stock Videos (10 per chunk — Pexels)</div>')
            gr.Markdown('10 videos downloaded per chunk keyword, cut together for a dynamic background.')
            btn_videos    = gr.Button('📹  Download Stock Videos', variant='primary')
            status_videos = gr.Textbox(label='Status', interactive=False, lines=1)

            gr.HTML('<hr style="margin:20px 0"/>')
            gr.HTML('<div class="step-label">④ Render Final Video (FFmpeg)</div>')
            gr.Markdown(
                'Multi-clip bg · 55% dark overlay · Character left · Karaoke captions right (gold word) · '
                'Optional bg music 12% · Output: 1920×1080 30fps H.264 MP4'
            )
            btn_render    = gr.Button('🎬  Render Final Video', variant='primary')
            status_render = gr.Textbox(label='Status', interactive=False, lines=2)
            output_video  = gr.Video(label='Final Video')

        # ── TAB 4: HELP ──────────────────────────────────────────────────
        with gr.Tab('❓  Help'):
            gr.Markdown('''
## Quick Start

### 0 — Optional: Extract a Formula First
Go to **Extract Formula** tab → paste an existing script → click Extract.
Formula is auto-saved. Then go to Studio tab and select it from the dropdown.

### 1 — Settings tab
- **Gemini API Key** — Free at aistudio.google.com
- **Pexels API Key** — Free at pexels.com/api
- **Voice Clone** — upload 3-10s clean voice → Clone & Save (one-time)
- **Character Image** — PNG transparent bg (optional)

### 2 — Studio tab — run steps 1→2→3→4
1. Select formula from dropdown (or paste in Settings → save → refresh)
2. Set chunks (3-20) and total script length (10K–50K chars)
3. Enter video title, pick language
4. Generate Script → Generate Audio → Download Videos → Render

### Architecture

```
EXTRACT FORMULA (gemini-2.5-pro, ~2-5 min):
  Script → PASS A (per-chunk patterns)
        → PASS B (synthesize Writing Guidelines)
        → PASS C (expand to 30K+ chars)
        → Auto-saved to formula library

GENERATE SCRIPT (gemini-2.5-flash, PARALLEL — all chunks at once):
  PHASE 1: Read formula → Extract DNA (10 law categories)
           → Build per-chunk recipes + Pexels keywords
           → DNA cached by MD5 (skip on re-run, saves 25s)
  PHASE 2: All chunks written in parallel (5 concurrent)
           → ~5× faster than sequential

STOCK VIDEOS: 10 videos per chunk keyword → cut together dynamically

GENERATE VIDEO:
  Voice Clone → TTS per chunk → Whisper word timestamps
  Pexels keywords → 10 clips each → multi-clip bg concat
  FFmpeg: multi-clip bg + dark overlay + character left + karaoke right
```
''')

    # ── EVENT HANDLERS (all at bottom to avoid forward-reference issues) ──

    clone_btn.click(handle_clone_voice, inputs=[voice_ref], outputs=[clone_status])

    save_formula_btn.click(
        handle_save_formula_manual,
        inputs=[save_formula_name, formula_box],
        outputs=[save_formula_status],
    )

    formula_dropdown.change(
        handle_formula_select,
        inputs=[formula_dropdown],
        outputs=[active_formula, formula_sel_status],
    )

    refresh_formula_btn.click(
        lambda: gr.Dropdown(choices=_get_formula_names()),
        outputs=[formula_dropdown],
    )

    fe_btn.click(
        handle_extract_formula,
        inputs=[gemini_key, fe_script, fe_name, fe_lang],
        outputs=[fe_output, fe_status, formula_dropdown],
    )

    btn_script.click(
        handle_generate_script,
        inputs=[gemini_key, active_formula, formula_box, num_chunks,
                total_script_chars, title_input, lang_select],
        outputs=[script_table, status_script],
    )

    btn_audio.click(handle_generate_audio, inputs=[lang_select], outputs=[status_audio])
    btn_videos.click(handle_download_videos, inputs=[pexels_key], outputs=[status_videos])
    btn_render.click(handle_render_video, outputs=[output_video, status_render])
    char_img.change(handle_set_character,   inputs=[char_img],     outputs=[char_status])
    bg_music_inp.change(handle_set_bgmusic, inputs=[bg_music_inp], outputs=[bgm_status])

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  LAUNCH (Gradio 6.x: theme + css go to launch(), not Blocks)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

demo.queue(default_concurrency_limit=1, max_size=20)
demo.launch(
    share=True,
    debug=True,
    show_error=True,
    inline=False,
    theme=gr.themes.Soft(),
    css=CSS,
)
print('\n🚀 Gradio UI is live!')
